In [ ]:
import pandas, statsmodels, matplotlib
print(pandas.__version__, statsmodels.__version__)

In [ ]:
import pandas as pd

files = ["train_sent_emo.csv", "dev_sent_emo.csv", "test_sent_emo.csv"]
parts = []
for f in files:
    d = pd.read_csv(f"../data/MELD/{f}")
    d["split"] = f.split("_")[0]
    parts.append(d)

df = pd.concat(parts, ignore_index=True)

print(df.shape)
print(df.columns.tolist())
df.head()

In [ ]:
# "00:16:16,059" -> තත්පර (comma එක තිතකට හරවලා)
df["start"] = pd.to_timedelta(df["StartTime"].str.replace(",", ".")).dt.total_seconds()
df["end"]   = pd.to_timedelta(df["EndTime"].str.replace(",", ".")).dt.total_seconds()
df["dur"]   = df["end"] - df["start"]

print("dur <= 0 වූ පේළි:", (df["dur"] <= 0).sum())
df["dur"].describe()

In [ ]:
# 1. The row with dur <= 0
print("Rows with dur <= 0:")
print(df[df["dur"] <= 0][["Utterance", "Speaker", "Dialogue_ID", "split", "StartTime", "EndTime", "dur"]])

# 2. Very long rows (more than 20 seconds)
long_rows = df[df["dur"] > 20][["Utterance", "Speaker", "Dialogue_ID", "split", "StartTime", "EndTime", "dur"]]
print("\nNumber of rows longer than 20 seconds:", len(long_rows))
print(long_rows.sort_values("dur", ascending=False).head(15).to_string())

# 3. Distribution of durations
print("\nDuration quantiles:")
print(df["dur"].quantile([0.5, 0.9, 0.99, 0.999]))

In [ ]:
MAX_DUR = 60  # seconds; rows above this are treated as timestamp errors

bad = (df["dur"] <= 0) | (df["dur"] > MAX_DUR)
print("Rows removed:", bad.sum(), f"({bad.mean():.3%})")
print(df[bad][["Utterance", "Speaker", "dur"]])

df_clean = df[~bad].copy()
print("Rows kept:", len(df_clean))
print(df_clean["dur"].describe())

In [ ]:
print(df_clean["Speaker"].value_counts().head(15))

gender_map = {
    "Chandler": "M", "Joey": "M", "Ross": "M",
    "Monica": "F", "Phoebe": "F", "Rachel": "F",
}
df_clean["gender"] = df_clean["Speaker"].map(gender_map)

print(df_clean["gender"].value_counts(dropna=False))
print("Unmapped share:", round(df_clean["gender"].isna().mean(), 3))

In [ ]:
d = df_clean.sort_values(["split", "Dialogue_ID", "Utterance_ID"])
prev_end = d.groupby(["split", "Dialogue_ID"])["end"].shift()
overlap = d["start"] < prev_end

print("Overlapping rows:", overlap.sum(), "out of", len(d))
print("Share:", round(overlap.mean(), 4))

In [ ]:
import os
os.makedirs("../outputs", exist_ok=True)

intervals = df_clean.dropna(subset=["gender"]).copy()
intervals["corpus"] = "MELD"
intervals = intervals.rename(columns={
    "Season": "season", "Episode": "episode",
    "Dialogue_ID": "dialogue_id", "Utterance_ID": "utterance_id",
    "Speaker": "speaker",
})[["corpus", "split", "season", "episode", "dialogue_id",
    "utterance_id", "speaker", "gender", "start", "end", "dur"]]

intervals.to_csv("../outputs/meld_intervals.csv", index=False)
print(intervals.shape)
intervals.head()

In [ ]:
g = (intervals.groupby(["season", "episode", "gender"])
     .agg(T=("dur", "sum"), N=("dur", "size"), mean_len=("dur", "mean"))
     .reset_index())

tot = g.groupby(["season", "episode"])[["T", "N"]].transform("sum")

g["speaking_time_share"] = g["T"] / tot["T"]
g["turn_count_share"]    = g["N"] / tot["N"]
g["mean_turn_length"]    = g["mean_len"] / (tot["T"] / tot["N"])

print(g.head(10).to_string())
print(g.groupby("gender")[["speaking_time_share", "turn_count_share", "mean_turn_length"]].mean())

In [ ]:
# Pooled shares over all principal-cast speech
pooled = intervals.groupby("gender")["dur"].agg(T="sum", N="size")
pooled["time_share"] = pooled["T"] / pooled["T"].sum()
pooled["turn_share"] = pooled["N"] / pooled["N"].sum()
print(pooled)

# Episodes where one gender is missing, and turns per episode
cnt = intervals.groupby(["season", "episode"])["gender"].nunique()
print("Episodes:", len(cnt), "| with only one gender:", (cnt < 2).sum())
print(intervals.groupby(["season", "episode"]).size().describe())

In [ ]:
MIN_TURNS = 20

ep = intervals.groupby(["season", "episode"]).agg(
    turns=("dur", "size"),
    genders=("gender", "nunique"),
)
keep = ep[(ep["turns"] >= MIN_TURNS) & (ep["genders"] == 2)].index

print("Episodes before:", len(ep))
print("Episodes kept  :", len(keep))

mask = intervals.set_index(["season", "episode"]).index.isin(keep)
shards = intervals[mask].copy()
print("Rows kept:", len(shards), "of", len(intervals))

In [ ]:
import numpy as np

def fragment(intervals, target="F", frac=0.5, k=3, gap=0.2, min_piece=0.3, seed=0):
    """Split a fraction `frac` of the target gender's eligible turns into k pieces.
    Total speaking time of every gender stays exactly the same."""
    rng = np.random.default_rng(seed)
    d = intervals.sort_values(["split", "dialogue_id", "utterance_id"]).reset_index(drop=True)

    # Room between this turn's end and the next turn's start in the same dialogue
    next_start = d.groupby(["split", "dialogue_id"])["start"].shift(-1)
    room = (next_start - d["end"]).fillna(np.inf)

    eligible = (d["gender"] == target) & (room >= (k - 1) * gap) & (d["dur"] >= k * min_piece)
    chosen = (eligible & (rng.random(len(d)) < frac)).to_numpy()

    out = []
    for i, r in enumerate(d.to_dict("records")):
        if not chosen[i]:
            out.append(r)
            continue
        piece = r["dur"] / k
        for j in range(k):
            p = dict(r)
            p["start"] = r["start"] + j * (piece + gap)
            p["end"] = p["start"] + piece
            p["dur"] = piece
            out.append(p)
    return pd.DataFrame(out)


def summary(df_):
    s = df_.groupby("gender")["dur"].agg(total_time="sum", turns="size", mean_len="mean")
    return s.round(3)

print("ORIGINAL")
print(summary(intervals))

aug = fragment(intervals, target="F", frac=0.5)
print("\nAFTER FRAGMENTATION (frac = 0.5)")
print(summary(aug))

# The key check: total speaking time must be unchanged for BOTH genders
orig_T = intervals.groupby("gender")["dur"].sum()
aug_T = aug.groupby("gender")["dur"].sum()
print("\nTotal time unchanged:", np.allclose(orig_T, aug_T))

In [ ]:
fracs = np.round(np.linspace(0.1, 1.0, 10), 1)
for f in fracs:
    a = fragment(intervals, target="F", frac=f)
    s = summary(a).loc["F"]
    print(f"frac={f}: F turns={int(s['turns'])}, F mean_len={s['mean_len']}, "
          f"time unchanged={np.allclose(intervals.groupby('gender')['dur'].sum(), a.groupby('gender')['dur'].sum())}")

In [ ]:
def shard_features(df):
    g = (df.groupby(["season", "episode", "gender"])
           .agg(T=("dur", "sum"), N=("dur", "size")).reset_index())
    g["mean_len"] = g["T"] / g["N"]
    w = g.pivot(index=["season", "episode"], columns="gender",
                values=["T", "N", "mean_len"]).dropna()
    out = pd.DataFrame(index=w.index)
    out["time_share_F"] = w["T"]["F"] / (w["T"]["F"] + w["T"]["M"])
    out["turn_share_F"] = w["N"]["F"] / (w["N"]["F"] + w["N"]["M"])
    out["len_ratio_F"]  = w["mean_len"]["F"] / w["mean_len"]["M"]
    return out

base = shards   # the 168-episode table from Cell 6b
base_F_turns = (base["gender"] == "F").sum()

rows = []
for f in [0.0] + list(np.round(np.linspace(0.1, 1.0, 10), 1)):
    a = base if f == 0 else fragment(base, target="F", frac=f)
    n_frag = ((a["gender"] == "F").sum() - base_F_turns) / 2   # k=3 -> each split adds 2 rows
    feats = shard_features(a).mean().round(4)
    rows.append({"frac": f,
                 "realised_share": round(n_frag / base_F_turns, 3),
                 **feats.to_dict()})

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd

def cluster(df, target="F", frac=0.5, seed=0):
    """Within each dialogue, move a fraction `frac` of the target gender's turns
    to the start of the dialogue. Durations and gaps are kept, so total speaking
    time, turn counts and mean turn length do not change."""
    rng = np.random.default_rng(seed)
    parts = []
    for _, d in df.sort_values(["split", "dialogue_id", "utterance_id"]).groupby(
            ["split", "dialogue_id"], sort=False):
        d = d.reset_index(drop=True)
        t0 = d["start"].min()
        gaps = (d["start"].shift(-1) - d["end"]).iloc[:-1].clip(lower=0).to_numpy()

        pick = ((d["gender"] == target) & (rng.random(len(d)) < frac)).to_numpy()
        order = np.concatenate([np.where(pick)[0], np.where(~pick)[0]])
        d = d.iloc[order].reset_index(drop=True)

        dur = d["dur"].to_numpy()
        starts = t0 + np.concatenate([[0.0], np.cumsum(dur[:-1] + gaps)])
        d["start"] = starts
        d["end"] = starts + dur
        d["utterance_id"] = np.arange(len(d))
        parts.append(d)
    return pd.concat(parts, ignore_index=True)


def dispersion_F(df, K=10, min_turns=3):
    """Normalised entropy (0..1) of F's speech over K equal time bins of each dialogue.
    High = spread out, low = bunched in one region. Averaged over dialogues
    where F has at least `min_turns` turns."""
    vals = []
    for _, d in df.groupby(["split", "dialogue_id"], sort=False):
        f = d[d["gender"] == "F"]
        if len(f) < min_turns:
            continue
        edges = np.linspace(d["start"].min(), d["end"].max(), K + 1)
        ov = (np.minimum(f["end"].to_numpy()[:, None], edges[1:])
              - np.maximum(f["start"].to_numpy()[:, None], edges[:-1])).clip(min=0).sum(axis=0)
        if ov.sum() == 0:
            continue
        p = ov / ov.sum()
        p = p[p > 0]
        vals.append(-(p * np.log(p)).sum() / np.log(K))
    return float(np.mean(vals))


base_T = shards.groupby("gender")["dur"].sum()
base_N = shards.groupby("gender").size()

rows = []
for f in [0.0] + list(np.round(np.linspace(0.1, 1.0, 10), 1)):
    a = cluster(shards, target="F", frac=f)
    feats = shard_features(a).mean().round(4)
    rows.append({
        "frac": f,
        **feats.to_dict(),
        "dispersion_F": round(dispersion_F(a), 4),
        "time_unchanged": bool(np.allclose(base_T, a.groupby("gender")["dur"].sum())),
        "turns_unchanged": bool((base_N == a.groupby("gender").size()).all()),
    })

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
def interrupt(df, target="F", frac=0.5, overlap_frac=0.3, seed=0):
    """For a fraction `frac` of (target turn -> other-gender turn) pairs, start the
    next turn `overlap_frac * dur` BEFORE the target turn ends. Durations and
    turn counts never change. Input must have no overlaps (gaps >= 0)."""
    rng = np.random.default_rng(seed)
    parts = []
    for _, d in df.sort_values(["split", "dialogue_id", "utterance_id"]).groupby(
            ["split", "dialogue_id"], sort=False):
        d = d.reset_index(drop=True)
        start = d["start"].to_numpy()
        end = d["end"].to_numpy()
        dur = d["dur"].to_numpy()
        gender = d["gender"].to_numpy()
        new_start = start.copy()
        shift = 0.0
        for j in range(len(d)):
            if j > 0 and gender[j - 1] == target and gender[j] != target and rng.random() < frac:
                gap = max(start[j] - end[j - 1], 0.0)
                shift += gap + overlap_frac * dur[j - 1]
            new_start[j] = start[j] - shift
        d["start"] = new_start
        d["end"] = new_start + dur
        parts.append(d)
    return pd.concat(parts, ignore_index=True)


def interruption_asym_F(df):
    """(times F was cut off - times F cut others off) / F turns, averaged over episodes."""
    d = df.sort_values(["split", "dialogue_id", "utterance_id"]).copy()
    grp = d.groupby(["split", "dialogue_id"], sort=False)
    nxt_start = grp["start"].shift(-1)
    nxt_gender = grp["gender"].shift(-1)
    ov = (nxt_start < d["end"] - 1e-9) & nxt_gender.notna() & (nxt_gender != d["gender"])
    d["cut_F"] = (ov & (d["gender"] == "F")).astype(int)   # F cut off by M
    d["cut_M"] = (ov & (d["gender"] == "M")).astype(int)   # M cut off by F (F cut others)
    d["is_F"] = (d["gender"] == "F").astype(int)
    e = d.groupby(["season", "episode"]).agg(cut_F=("cut_F", "sum"),
                                              cut_M=("cut_M", "sum"),
                                              nF=("is_F", "sum"))
    return float(((e["cut_F"] - e["cut_M"]) / e["nF"]).mean())


base0 = cluster(shards, target="F", frac=0.0)       # overlap-free baseline
base_T = base0.groupby("gender")["dur"].sum()
base_N = base0.groupby("gender").size()

rows = []
for f in [0.0] + list(np.round(np.linspace(0.1, 1.0, 10), 1)):
    a = base0 if f == 0 else interrupt(base0, target="F", frac=f)
    feats = shard_features(a).mean().round(4)
    rows.append({
        "frac": f,
        **feats.to_dict(),
        "interruption_asym_F": round(interruption_asym_F(a), 4),
        "dispersion_F": round(dispersion_F(a), 4),
        "time_unchanged": bool(np.allclose(base_T, a.groupby("gender")["dur"].sum())),
        "turns_unchanged": bool((base_N == a.groupby("gender").size()).all()),
    })

print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
from scipy.stats import spearmanr

def dispersion_by_episode(df, K=10, min_turns=3):
    vals = {}
    for _, d in df.groupby(["split", "dialogue_id"], sort=False):
        f = d[d["gender"] == "F"]
        if len(f) < min_turns:
            continue
        edges = np.linspace(d["start"].min(), d["end"].max(), K + 1)
        ov = (np.minimum(f["end"].to_numpy()[:, None], edges[1:])
              - np.maximum(f["start"].to_numpy()[:, None], edges[:-1])).clip(min=0).sum(axis=0)
        if ov.sum() == 0:
            continue
        p = ov / ov.sum()
        p = p[p > 0]
        key = (d["season"].iloc[0], d["episode"].iloc[0])
        vals.setdefault(key, []).append(-(p * np.log(p)).sum() / np.log(K))
    return pd.Series({k: float(np.mean(v)) for k, v in vals.items()})


def episode_features(df):
    d = df.sort_values(["split", "dialogue_id", "utterance_id"]).copy()
    out = shard_features(d)                      # time_share_F, turn_share_F, len_ratio_F

    grp = d.groupby(["split", "dialogue_id"], sort=False)
    nxt_start = grp["start"].shift(-1)
    nxt_gender = grp["gender"].shift(-1)
    ov = (nxt_start < d["end"] - 1e-9) & nxt_gender.notna() & (nxt_gender != d["gender"])
    d["cut_F"] = (ov & (d["gender"] == "F")).astype(int)
    d["cut_M"] = (ov & (d["gender"] == "M")).astype(int)
    d["is_F"] = (d["gender"] == "F").astype(int)
    e = d.groupby(["season", "episode"]).agg(cut_F=("cut_F", "sum"),
                                              cut_M=("cut_M", "sum"),
                                              nF=("is_F", "sum"))
    out["interruption_asym_F"] = ((e["cut_F"] - e["cut_M"]) / e["nF"]).reindex(out.index)
    out["dispersion_F"] = dispersion_by_episode(d).reindex(out.index)
    return out


ops = {
    "O1_fragment":  lambda d, f: fragment(d, target="F", frac=f),
    "O2_interrupt": lambda d, f: interrupt(d, target="F", frac=f),
    "O3_cluster":   lambda d, f: cluster(d, target="F", frac=f),
}
levels = np.round(np.linspace(0.0, 1.0, 11), 1)

frames = []
for name, fn in ops.items():
    for f in levels:
        a = base0 if f == 0 else fn(base0, f)
        feats = episode_features(a).reset_index()
        feats["operator"] = name
        feats["level"] = f
        frames.append(feats)

aug_table = pd.concat(frames, ignore_index=True)
aug_table.to_csv("../outputs/meld_augmented_shards.csv", index=False)

feat_cols = ["time_share_F", "turn_share_F", "len_ratio_F",
             "interruption_asym_F", "dispersion_F"]

res = {}
for name in ops:
    sub = aug_table[aug_table["operator"] == name]
    res[name] = {c: round(spearmanr(sub["level"], sub[c], nan_policy="omit")[0], 3)
                 for c in feat_cols}

print("Table shape:", aug_table.shape)
print("\nShare of missing values per feature:")
print(aug_table[feat_cols].isna().mean().round(3))
print("\nSpearman correlation between injected level and each feature:")
print(pd.DataFrame(res).T)

In [ ]:
def dispersion_by_episode_g(df, gender, K=10, min_turns=3):
    vals = {}
    for _, d in df.groupby(["split", "dialogue_id"], sort=False):
        g = d[d["gender"] == gender]
        if len(g) < min_turns:
            continue
        edges = np.linspace(d["start"].min(), d["end"].max(), K + 1)
        ov = (np.minimum(g["end"].to_numpy()[:, None], edges[1:])
              - np.maximum(g["start"].to_numpy()[:, None], edges[:-1])).clip(min=0).sum(axis=0)
        if ov.sum() == 0:
            continue
        p = ov / ov.sum()
        p = p[p > 0]
        key = (d["season"].iloc[0], d["episode"].iloc[0])
        vals.setdefault(key, []).append(-(p * np.log(p)).sum() / np.log(K))
    return pd.Series({k: float(np.mean(v)) for k, v in vals.items()})


def episode_features_signed(df):
    out = episode_features(df)   # from Cell 12
    out["dispersion_M"] = dispersion_by_episode_g(df, "M").reindex(out.index)
    out["dispersion_diff"] = out["dispersion_F"] - out["dispersion_M"]
    out["log_len_ratio"] = np.log(out["len_ratio_F"])
    return out


ops2 = {
    "O1_fragment":  lambda d, t, f: fragment(d, target=t, frac=f),
    "O2_interrupt": lambda d, t, f: interrupt(d, target=t, frac=f),
    "O3_cluster":   lambda d, t, f: cluster(d, target=t, frac=f),
}
levels = np.round(np.linspace(0.1, 1.0, 10), 1)

b = episode_features_signed(base0).reset_index()
b["operator"] = "baseline"
b["signed_level"] = 0.0
frames = [b]

for name, fn in ops2.items():
    for tgt, sign in [("F", 1), ("M", -1)]:
        for f in levels:
            feats = episode_features_signed(fn(base0, tgt, f)).reset_index()
            feats["operator"] = name
            feats["signed_level"] = sign * f
            frames.append(feats)

sdf = pd.concat(frames, ignore_index=True)
sdf.to_csv("../outputs/meld_augmented_signed.csv", index=False)

print("Shape:", sdf.shape)
print(sdf[["time_share_F", "turn_share_F", "log_len_ratio",
           "interruption_asym_F", "dispersion_diff"]].isna().mean().round(3))
print(sdf.groupby("operator")[["turn_share_F", "log_len_ratio",
                                "interruption_asym_F", "dispersion_diff"]].mean().round(3))

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV

feature_cols = ["time_share_F", "turn_share_F", "log_len_ratio",
                "interruption_asym_F", "dispersion_diff"]

data = sdf.dropna(subset=feature_cols).copy()
data["label"] = 10 * data["signed_level"]          # provisional label, see note below
groups = data["season"].astype(str) + "_" + data["episode"].astype(str)
print("Rows used:", len(data), "| episodes:", groups.nunique())

def cv_report(cols, name):
    X, y = data[cols].to_numpy(), data["label"].to_numpy()
    pipe = make_pipeline(StandardScaler(), PolynomialFeatures(include_bias=False), Ridge())
    grid = {"polynomialfeatures__degree": [1, 2], "ridge__alpha": [0.1, 1, 10, 100]}
    gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=5), scoring="r2")
    gs.fit(X, y, groups=groups)
    print(f"{name:28s} CV R2 = {gs.best_score_:.3f}   best = {gs.best_params_}")

cv_report(feature_cols,            "All 5 features")
cv_report(["time_share_F"],        "B1: speaking time only")
cv_report(["turn_share_F"],        "B2: turn count only")

In [ ]:
from sklearn.model_selection import cross_val_predict
from scipy.stats import spearmanr

X = data[feature_cols].to_numpy()
y = data["label"].to_numpy()

pipe = make_pipeline(StandardScaler(),
                     PolynomialFeatures(degree=2, include_bias=False),
                     Ridge(alpha=100))
data["pred"] = cross_val_predict(pipe, X, y, cv=GroupKFold(n_splits=5), groups=groups)

print("Spearman(predicted score, injected level) per operator:")
for name, sub in data[data["operator"] != "baseline"].groupby("operator"):
    print(f"  {name:14s} {spearmanr(sub['pred'], sub['signed_level'])[0]:.3f}")

base_pred = data.loc[data["operator"] == "baseline", "pred"]
print("\nBaseline episodes (no injected bias):")
print("  predicted score mean =", round(base_pred.mean(), 3),
      "| std =", round(base_pred.std(), 3))

print("\nMean predicted score by signed level (all operators pooled):")
print(data.groupby("signed_level")["pred"].mean().round(2).to_string())

In [ ]:
def position_gap_by_episode(df):
    """Centroid of F speech minus centroid of M speech (0..1 position inside each dialogue)."""
    vals = {}
    for _, d in df.groupby(["split", "dialogue_id"], sort=False):
        t0, t1 = d["start"].min(), d["end"].max()
        span = t1 - t0
        if span <= 0:
            continue
        mid = ((d["start"] + d["end"]) / 2 - t0) / span
        w = d["dur"]
        cen = {}
        for g in ["F", "M"]:
            m = d["gender"] == g
            if m.sum() == 0 or w[m].sum() == 0:
                break
            cen[g] = float((mid[m] * w[m]).sum() / w[m].sum())
        else:
            key = (d["season"].iloc[0], d["episode"].iloc[0])
            vals.setdefault(key, []).append(cen["F"] - cen["M"])
    return pd.Series({k: float(np.mean(v)) for k, v in vals.items()})


recs = [pd.DataFrame({
    "signed_level": 0.0,
    "position_gap": position_gap_by_episode(base0),
    "disp_F": dispersion_by_episode_g(base0, "F"),
    "disp_M": dispersion_by_episode_g(base0, "M"),
})]
for tgt, sign in [("F", 1), ("M", -1)]:
    for f in np.round(np.linspace(0.1, 1.0, 10), 1):
        a = cluster(base0, target=tgt, frac=f)
        recs.append(pd.DataFrame({
            "signed_level": sign * f,
            "position_gap": position_gap_by_episode(a),
            "disp_F": dispersion_by_episode_g(a, "F"),
            "disp_M": dispersion_by_episode_g(a, "M"),
        }))

o3 = pd.concat(recs).dropna()
print(o3.groupby("signed_level")[["position_gap", "disp_F", "disp_M"]].mean().round(3).to_string())

print("\nSpearman with injected level:")
for c in ["position_gap", "disp_F", "disp_M"]:
    print(f"  {c:13s} {spearmanr(o3['signed_level'], o3[c])[0]:.3f}")

In [ ]:
def pg_table(d, operator, signed_level):
    s = (position_gap_by_episode(d)
         .rename_axis(["season", "episode"]).rename("position_gap").reset_index())
    s["operator"] = operator
    s["signed_level"] = signed_level
    return s

parts = [pg_table(base0, "baseline", 0.0)]
for name, fn in ops2.items():
    for tgt, sign in [("F", 1), ("M", -1)]:
        for f in np.round(np.linspace(0.1, 1.0, 10), 1):
            parts.append(pg_table(fn(base0, tgt, f), name, sign * f))

pg = pd.concat(parts, ignore_index=True)
data2 = sdf.merge(pg, on=["season", "episode", "operator", "signed_level"], how="left")

feature_cols2 = ["time_share_F", "turn_share_F", "log_len_ratio",
                 "interruption_asym_F", "position_gap"]
data2 = data2.dropna(subset=feature_cols2).copy()
data2["label"] = 10 * data2["signed_level"]
groups2 = data2["season"].astype(str) + "_" + data2["episode"].astype(str)
X2, y2 = data2[feature_cols2].to_numpy(), data2["label"].to_numpy()

pipe = make_pipeline(StandardScaler(), PolynomialFeatures(include_bias=False), Ridge())
grid = {"polynomialfeatures__degree": [1, 2], "ridge__alpha": [1, 10, 100, 1000]}
gs = GridSearchCV(pipe, grid, cv=GroupKFold(n_splits=5), scoring="r2")
gs.fit(X2, y2, groups=groups2)
print("Rows:", len(data2), "| CV R2 =", round(gs.best_score_, 3), gs.best_params_)

data2["pred"] = cross_val_predict(gs.best_estimator_, X2, y2,
                                  cv=GroupKFold(n_splits=5), groups=groups2)
for name, sub in data2[data2["operator"] != "baseline"].groupby("operator"):
    print(f"{name:14s} Spearman = {spearmanr(sub['pred'], sub['signed_level'])[0]:.3f}")
b = data2.loc[data2["operator"] == "baseline", "pred"]
print("Baseline score mean / std:", round(b.mean(), 3), round(b.std(), 3))

In [ ]:
data2.to_csv("../outputs/meld_augmented_model_table.csv", index=False)
print("Episodes used:", groups2.nunique(), "| rows:", len(data2))
print(data2.groupby("signed_level")["pred"].mean().round(2).to_string())

In [ ]:
data2.to_csv("../outputs/meld_augmented_model_table.csv", index=False)
print("Episodes used:", groups2.nunique(), "| rows:", len(data2))
print(data2.groupby("signed_level")["pred"].mean().round(2).to_string())

In [ ]:
import pandas as pd
mi = pd.read_csv("../outputs/meld_intervals.csv")
print(mi.shape)
print(list(mi.columns))
print(mi.head(3).to_string())
print(mi["gender"].value_counts().to_dict() if "gender" in mi.columns else "no gender column")

In [ ]:
print(list(df.columns))

In [ ]:
import re

def count_words(s):
    s = re.sub(r"\([^)]*\)?", " ", str(s))
    return len(re.findall(r"[A-Za-z0-9']+", s))

tmp = df[["split", "Dialogue_ID", "Utterance_ID", "Utterance"]].copy()
tmp["n_words"] = tmp["Utterance"].map(count_words)

mi = pd.read_csv("../outputs/meld_intervals.csv")
n0 = len(mi)
mi = mi.merge(tmp[["split", "Dialogue_ID", "Utterance_ID", "n_words"]],
              left_on=["split", "dialogue_id", "utterance_id"],
              right_on=["split", "Dialogue_ID", "Utterance_ID"], how="left")
mi = mi.drop(columns=["Dialogue_ID", "Utterance_ID"])
print("rows before/after merge:", n0, len(mi))
print("rows without word count:", int(mi["n_words"].isna().sum()))
print("share of utterances with <= 2 words: %.3f" % (mi["n_words"] <= 2).mean())
mi.to_csv("../outputs/meld_intervals_words.csv", index=False)

In [ ]:
meld = pd.read_csv("../outputs/meld_intervals_words.csv")
meld["wps"] = meld["n_words"] / meld["dur"]
bad = (meld["dur"] > 8) & (meld["wps"] < 0.5)
tot = meld["dur"].sum()
print("suspicious intervals (dur > 8 s and < 0.5 words/s):", int(bad.sum()),
      "| share of rows:", round(float(bad.mean()), 4),
      "| share of speaking time:", round(float(meld.loc[bad, "dur"].sum() / tot), 4))
print("by gender (rows):", meld.loc[bad, "gender"].value_counts().to_dict())
print("by gender (seconds):", meld.loc[bad].groupby("gender")["dur"].sum().round(1).to_dict())

m = meld.sort_values(["season", "episode", "dialogue_id", "speaker", "start"])
prev_end = m.groupby(["season", "episode", "dialogue_id", "speaker"])["end"].shift()
ovl = (prev_end - m["start"]).clip(lower=0).fillna(0)
print("overlap seconds:", round(float(ovl.sum()), 1), "| share of speaking time:", round(float(ovl.sum() / tot), 4))
print("overlap by gender (s):", ovl.groupby(m["gender"]).sum().round(1).to_dict())